# Refining the prompt to get the desired output

The next step is to get closer to the desired output by refining the prompt given to the agents.  The more lengthy prompt is defined in instructions.py.  The agent will leverage it capability to perform web search as well as save and retrieve files from disk to produce output that is more useful for planning a vacation.


In [1]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool
from agents.mcp import MCPServerStdio
import instructions

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


In [2]:
# Print out the more details instructions for the agent

print(instructions.trip_planner_instructions)

You are a methodical and detail-oriented trip planning assistant. Your task is to create a COMPLETE, timeline-based trip itinerary with specific departure/arrival times and durations for every activity.

CRITICAL: You must complete the ENTIRE itinerary before finishing. Do not stop at research phase. Do not ask for permission to continue. Work through all steps until you have a fully detailed day-by-day schedule.

The customer has provided the following details for their trip:
- Home Location: Columbus, OH
- Departure Date: 2026/02/25
- Return Date: 2026/03/07
- Destination: Tokyo, Japan
- Must-Do Activities: Visit the Tokyo Tower, Explore Akihabara, Experience a traditional tea ceremony, Visit the Tsukiji Fish Market, Take a day trip to Mount Fuji   
- Number of Travelers: 3
- Ages of Travelers: 51, 50, 17
- Other Considerations: 
  - I will be running in the Tokyo marathon on Sunday March 1, so I only need a relaxing place to eat on that day.
  - Starting on March 3, throughout the r

In [3]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
filesystem_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

web_search_tool = WebSearchTool(search_context_size="low")

async with MCPServerStdio(params=filesystem_params, client_session_timeout_seconds=30) as mcp_server_files:
    trip_planner_agent = Agent(
        name="Trip Planner Agent",
        instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
        tools=[web_search_tool],
        mcp_servers=[mcp_server_files]
    )
    with trace("Trip Planner Agent"):
        result = await Runner.run(trip_planner_agent, instructions.trip_planner_instructions)
        print(result.final_output)

Your full, detailed, hour-by-hour trip itinerary—covering all flights, trains, subway routes, attraction visits, meals, and costs—for your Columbus, OH to Tokyo, Kyoto, and Osaka journey from Feb 25 to Mar 7, 2026 has been completed and saved as "trip_plan_using_detailed_instructions.md".

This file contains everything you need for a seamless trip: exact timing, transit lines, must-do experiences, meal recommendations with locations and costs, and a total day-by-day cost breakdown.

If you need further changes (e.g., alternative routing returning from Osaka/Kyoto) or want a PDF version, just ask! Safe travels!
